# Alpamayo 1.5: Offline Sliding-Window Evaluation

This notebook runs sliding-window offline inference over a whole local clip.

This version:
- builds clip cache once
- reuses the same evaluation function as the single-window notebook
- keeps the currently validated eval-side xy rotation pipeline


In [ ]:
import os
import sys
from pathlib import Path

repo_root = Path.cwd().resolve().parent
src_path = repo_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print('cwd =', Path.cwd().resolve())
print('repo_root =', repo_root)
print('src_path =', src_path)
print('src exists =', src_path.exists())
print('PYTHONPATH =', os.environ.get('PYTHONPATH'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from transformers import BitsAndBytesConfig

from alpamayo1_5 import helper
from alpamayo1_5.load_offline_dataset import (
    load_offline_dataset,
    get_or_build_offline_clip_cache,
)
from alpamayo1_5.models.alpamayo1_5 import Alpamayo1_5
from alpamayo1_5.offline_eval_utils import (
    set_reproducible_seeds,
    run_offline_inference_window,
    _compute_adaptive_xy_limits,
)


In [ ]:
SEED = 42
set_reproducible_seeds(SEED)
print('SEED =', SEED)


In [ ]:
# ===== Local paths =====
CLIP_DIR = Path('/workspace/dataset/')
MODEL_PATH = Path('/root/.cache/huggingface/hub/models--nvidia--Alpamayo-1.5-10B/snapshots/f11cd25b758ab560114019b555dde2a8b92d88b4')
PROCESSOR_PATH = Path('/root/.cache/huggingface/hub/models--Qwen--Qwen3-VL-8B-Instruct/snapshots/0c351dd01ed87e9c1b53cbc748cba10e6187ff3b')
COSMOS_PROCESSOR_PATH = Path('/root/.cache/huggingface/hub/models--nvidia--Cosmos-Reason2-8B/snapshots/f715d875a8a99a0a2b65aa28633afd9417e46bd9')

# ===== Inference config =====
DEVICE = 'cuda'
NUM_HISTORY_STEPS = 16
NUM_FUTURE_STEPS = 64
TIME_STEP = 0.1
NUM_FRAMES = 4
FPS = 30.0
FRAME0_GPS_TIME_SOD = 175484.98
NUM_TRAJ_SAMPLES = 1
TEMPERATURE = 0.6
TOP_P = 0.98
MAX_GENERATION_LENGTH = 256
IMAGE_SIZE = (448, 800)

# ===== Sliding config =====
SLIDING_START_T0_SOD = 175490.23
SLIDING_END_T0_SOD = 175506.23
SLIDING_STEP_SEC = 1.0
NUM_PLOT_WINDOWS = 6

# Keep the currently validated plotting/eval convention
EVAL_XY_ROTATION_DEG = -90.0

os.environ['ALPAMAYO_VLM_PROCESSOR_PATH'] = str(COSMOS_PROCESSOR_PATH)

print('CLIP_DIR =', CLIP_DIR)
print('MODEL_PATH =', MODEL_PATH)
print('PROCESSOR_PATH =', PROCESSOR_PATH)
print('COSMOS_PROCESSOR_PATH =', COSMOS_PROCESSOR_PATH)
print('ALPAMAYO_VLM_PROCESSOR_PATH =', os.environ['ALPAMAYO_VLM_PROCESSOR_PATH'])
print('SLIDING_START_T0_SOD =', SLIDING_START_T0_SOD)
print('SLIDING_END_T0_SOD =', SLIDING_END_T0_SOD)
print('SLIDING_STEP_SEC =', SLIDING_STEP_SEC)
print('EVAL_XY_ROTATION_DEG =', EVAL_XY_ROTATION_DEG)
print('IMAGE_SIZE =', IMAGE_SIZE)


### Load model and processor


In [ ]:
quantization_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,
    llm_int8_enable_fp32_cpu_offload=False,
)

model = Alpamayo1_5.from_pretrained(
    str(MODEL_PATH),
    dtype=torch.bfloat16,
    device_map='cuda:0',
    quantization_config=quantization_config,
)

processor = helper.get_processor(
    model.tokenizer,
    processor_name_or_path=PROCESSOR_PATH,
)

print('Model and processor loaded.')


### Build clip cache once


In [ ]:
clip_cache = get_or_build_offline_clip_cache(CLIP_DIR, debug=True, force_rebuild=False)
print('clip cache ready')
print('num cameras =', len(clip_cache.camera_paths))
print('num pose samples =', len(clip_cache.pose_times_sod))


In [ ]:
def compute_t0_grid(start_sod, end_sod, step_sec):
    n = int(np.floor((end_sod - start_sod) / step_sec)) + 1
    return start_sod + np.arange(n, dtype=np.float64) * step_sec


### Run sliding-window evaluation


In [ ]:
t0_grid = compute_t0_grid(
    start_sod=SLIDING_START_T0_SOD,
    end_sod=SLIDING_END_T0_SOD,
    step_sec=SLIDING_STEP_SEC,
)

print('num_windows =', len(t0_grid))
print('t0_grid[:5] =', t0_grid[:5].tolist())
print('t0_grid[-5:] =', t0_grid[-5:].tolist())

rows = []
viz_samples = []

for idx, t0_sod in enumerate(t0_grid):
    try:
        data = load_offline_dataset(
            clip_dir=CLIP_DIR,
            t0_sod=float(t0_sod),
            num_history_steps=NUM_HISTORY_STEPS,
            num_future_steps=NUM_FUTURE_STEPS,
            time_step=TIME_STEP,
            num_frames=NUM_FRAMES,
            fps=FPS,
            frame0_gps_time_sod=FRAME0_GPS_TIME_SOD,
            debug=False,
            image_size=IMAGE_SIZE,
        )

        result = run_offline_inference_window(
            data=data,
            model=model,
            processor=processor,
            device=DEVICE,
            top_p=TOP_P,
            temperature=TEMPERATURE,
            num_traj_samples=NUM_TRAJ_SAMPLES,
            max_generation_length=MAX_GENERATION_LENGTH,
            time_step=TIME_STEP,
            eval_xy_rotation_deg=EVAL_XY_ROTATION_DEG,
        )

        requested_idx_np = data['requested_video_frame_indices'].cpu().numpy()
        actual_idx_np = data['actual_video_frame_indices'].cpu().numpy()

        row = {
            'window_idx': idx,
            't0_sod': float(t0_sod),
            'best_sample_idx': int(result['df_metrics'].iloc[0]['best_sample_idx']),
            'minADE_m': float(result['df_metrics'].iloc[0]['minADE_m']),
            'final_point_error_m': float(result['df_metrics'].iloc[0]['final_point_error_m']),
            'gt_mean_speed_mps': float(result['df_metrics'].iloc[0]['gt_mean_speed_mps']),
            'pred_mean_speed_mps': float(result['df_metrics'].iloc[0]['pred_mean_speed_mps']),
            'speed_error_mps': float(result['df_metrics'].iloc[0]['speed_error_mps']),
            'gt_yaw_deg': float(result['df_metrics'].iloc[0]['gt_yaw_deg']),
            'pred_yaw_deg': float(result['df_metrics'].iloc[0]['pred_yaw_deg']),
            'yaw_error_deg': float(result['df_metrics'].iloc[0]['yaw_error_deg']),
            'mean_abs_yaw_minus_dxy_deg': float(result['history_diag']['mean_abs_yaw_minus_dxy_deg']),
            'last5_mean_abs_yaw_minus_dxy_deg': float(result['history_diag']['last5_mean_abs_yaw_minus_dxy_deg']),
            'num_clipped_frames_total': int(data['num_clipped_frames_total']),
            'requested_frame_min': int(requested_idx_np.min()),
            'requested_frame_max': int(requested_idx_np.max()),
            'actual_frame_min': int(actual_idx_np.min()),
            'actual_frame_max': int(actual_idx_np.max()),
            'pose_time_start_sod': float(data['pose_time_range_sod'][0]),
            'pose_time_end_sod': float(data['pose_time_range_sod'][1]),
            'history_time_start_sod': float(data['history_time_range_sod'][0]),
            'history_time_end_sod': float(data['history_time_range_sod'][1]),
            'future_time_start_sod': float(data['future_time_range_sod'][0]),
            'future_time_end_sod': float(data['future_time_range_sod'][1]),
            'image_time_start_sod': float(data['image_time_range_sod'][0]),
            'image_time_end_sod': float(data['image_time_range_sod'][1]),
            'pose_margin_left_sec': float(data['time_alignment_summary']['pose_margin_left_sec']),
            'pose_margin_right_sec': float(data['time_alignment_summary']['pose_margin_right_sec']),
            'cot': str(result['cot_value']),
            'status': 'ok',
            'error': '',
        }
        rows.append(row)

        viz_samples.append({
            't0_sod': float(t0_sod),
            'hist_xyz_plot': result['hist_xyz_plot'],
            'gt_xyz_plot': result['gt_xyz_plot'],
            'pred_best_xyz': result['pred_best_xyz'],
            'minADE_m': float(result['df_metrics'].iloc[0]['minADE_m']),
            'final_point_error_m': float(result['df_metrics'].iloc[0]['final_point_error_m']),
        })

        print(
            f"[{idx+1}/{len(t0_grid)}] t0={t0_sod:.2f} ok | "
            f"minADE={result['df_metrics'].iloc[0]['minADE_m']:.3f} | "
            f"final={result['df_metrics'].iloc[0]['final_point_error_m']:.3f} | "
            f"clipped={int(data['num_clipped_frames_total'])}"
        )

    except Exception as e:
        rows.append({
            'window_idx': idx,
            't0_sod': float(t0_sod),
            'best_sample_idx': np.nan,
            'minADE_m': np.nan,
            'final_point_error_m': np.nan,
            'gt_mean_speed_mps': np.nan,
            'pred_mean_speed_mps': np.nan,
            'speed_error_mps': np.nan,
            'gt_yaw_deg': np.nan,
            'pred_yaw_deg': np.nan,
            'yaw_error_deg': np.nan,
            'mean_abs_yaw_minus_dxy_deg': np.nan,
            'last5_mean_abs_yaw_minus_dxy_deg': np.nan,
            'num_clipped_frames_total': np.nan,
            'requested_frame_min': np.nan,
            'requested_frame_max': np.nan,
            'actual_frame_min': np.nan,
            'actual_frame_max': np.nan,
            'pose_time_start_sod': np.nan,
            'pose_time_end_sod': np.nan,
            'history_time_start_sod': np.nan,
            'history_time_end_sod': np.nan,
            'future_time_start_sod': np.nan,
            'future_time_end_sod': np.nan,
            'image_time_start_sod': np.nan,
            'image_time_end_sod': np.nan,
            'pose_margin_left_sec': np.nan,
            'pose_margin_right_sec': np.nan,
            'cot': '',
            'status': 'error',
            'error': repr(e),
        })
        print(f'[{idx+1}/{len(t0_grid)}] t0={t0_sod:.2f} ERROR | {repr(e)}')

df_results = pd.DataFrame(rows)
df_ok = df_results[df_results['status'] == 'ok'].reset_index(drop=True)

print('\n[Sliding evaluation results]')
display(df_results)

print('\n[Successful windows]')
display(df_ok)


### Time-alignment diagnostics across windows


In [ ]:
print('[Windows with frame clipping]')
display(
    df_ok[df_ok['num_clipped_frames_total'] > 0][[
        'window_idx',
        't0_sod',
        'num_clipped_frames_total',
        'requested_frame_min',
        'requested_frame_max',
        'actual_frame_min',
        'actual_frame_max',
        'pose_margin_left_sec',
        'pose_margin_right_sec',
        'minADE_m',
        'final_point_error_m',
    ]].sort_values(['num_clipped_frames_total', 't0_sod'], ascending=[False, True])
)

print('[Windows closest to pose-range boundary]')
display(
    df_ok[[
        'window_idx',
        't0_sod',
        'pose_margin_left_sec',
        'pose_margin_right_sec',
        'num_clipped_frames_total',
        'minADE_m',
        'final_point_error_m',
    ]].sort_values(['pose_margin_left_sec', 'pose_margin_right_sec'], ascending=[True, True]).head(15)
)


### Summary statistics


In [ ]:
if len(df_ok) == 0:
    raise RuntimeError('No successful sliding windows. Please inspect df_results.')

summary = pd.DataFrame([{
    'num_windows_total': int(len(df_results)),
    'num_windows_ok': int(len(df_ok)),
    'success_rate': float(len(df_ok) / len(df_results)),
    'minADE_mean_m': float(df_ok['minADE_m'].mean()),
    'minADE_median_m': float(df_ok['minADE_m'].median()),
    'minADE_p90_m': float(df_ok['minADE_m'].quantile(0.90)),
    'final_error_mean_m': float(df_ok['final_point_error_m'].mean()),
    'final_error_median_m': float(df_ok['final_point_error_m'].median()),
    'final_error_p90_m': float(df_ok['final_point_error_m'].quantile(0.90)),
    'abs_speed_error_mean_mps': float(df_ok['speed_error_mps'].abs().mean()),
    'abs_yaw_error_mean_deg': float(df_ok['yaw_error_deg'].abs().mean()),
    'hist_consistency_mean_deg': float(df_ok['mean_abs_yaw_minus_dxy_deg'].mean()),
    'mean_num_clipped_frames_total': float(df_ok['num_clipped_frames_total'].mean()),
}])

print('[Sliding summary]')
display(summary)

print('[Worst windows by minADE]')
display(df_ok.sort_values('minADE_m', ascending=False).head(10))

print('[Worst windows by final_point_error_m]')
display(df_ok.sort_values('final_point_error_m', ascending=False).head(10))


### Metric trends over time


In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(12, 15), sharex=True)

axes[0].plot(df_ok['t0_sod'], df_ok['minADE_m'], 'o-', label='minADE_m')
axes[0].set_ylabel('minADE (m)')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(df_ok['t0_sod'], df_ok['final_point_error_m'], 'o-', label='final_point_error_m', color='tab:orange')
axes[1].set_ylabel('Final Error (m)')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

axes[2].plot(df_ok['t0_sod'], df_ok['gt_mean_speed_mps'], 'o-', label='GT mean speed')
axes[2].plot(df_ok['t0_sod'], df_ok['pred_mean_speed_mps'], 'o-', label='Pred mean speed')
axes[2].set_ylabel('Speed (m/s)')
axes[2].grid(True, alpha=0.3)
axes[2].legend()

axes[3].plot(df_ok['t0_sod'], df_ok['num_clipped_frames_total'], 'o-', label='num_clipped_frames_total', color='tab:red')
axes[3].set_xlabel('t0_sod')
axes[3].set_ylabel('Clipped Frames')
axes[3].grid(True, alpha=0.3)
axes[3].legend()

plt.tight_layout()
plt.show()


### Plot sampled windows


In [ ]:
if len(viz_samples) == 0:
    raise RuntimeError('No successful windows to visualize.')

plot_indices = np.linspace(0, len(viz_samples) - 1, min(NUM_PLOT_WINDOWS, len(viz_samples)), dtype=int)

for plot_rank, i in enumerate(plot_indices):
    sample = viz_samples[int(i)]
    hist_xyz_plot = sample['hist_xyz_plot']
    gt_xyz_plot = sample['gt_xyz_plot']
    pred_best_xyz = sample['pred_best_xyz']

    xlim, ylim = _compute_adaptive_xy_limits(hist_xyz_plot, gt_xyz_plot, pred_best_xyz)

    plt.figure(figsize=(6, 6))
    plt.plot(hist_xyz_plot[:, 0], hist_xyz_plot[:, 1], 'b-o', linewidth=2, markersize=3, label='History')
    plt.plot(gt_xyz_plot[:, 0], gt_xyz_plot[:, 1], 'k-o', linewidth=2.5, markersize=3.5, label='GT')
    plt.plot(pred_best_xyz[:, 0], pred_best_xyz[:, 1], 'r-o', linewidth=2.5, markersize=3.5, label='Pred')
    plt.scatter([0.0], [0.0], c='red', marker='x', s=120, label='t0 ego')
    plt.xlabel('x')
    plt.ylabel('y')
    plt.title(
        f"Window {plot_rank+1}/{len(plot_indices)} | t0={sample['t0_sod']:.2f}\n"
        f"minADE={sample['minADE_m']:.3f}m, final={sample['final_point_error_m']:.3f}m"
    )
    plt.xlim(*xlim)
    plt.ylim(*ylim)
    plt.grid(True, alpha=0.3)
    plt.gca().set_aspect('equal', adjustable='box')
    plt.legend()
    plt.tight_layout()
    plt.show()
